In [2]:
%pip install pandas numpy matplotlib seaborn spacy scikit-learn wordcloud textstat lexical_diversity sentence_transformers transformers xgboost shap lime ipywidgets joblib streamlit lexicalrichness

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lexicalrichness: filename=lexicalrichness-0.5.1-py3-none-any.whl size=15418 sha256=0a235e61207fd3859c170f20c380b3860df4e47a323ff2e2c5d3959abe4c323a
  Stored in directory: /root/.cache/pip/wheels/eb/40/d0/053edb84485f223effdbf0f91fc2b6ec6fc6cf2230aadca09a
Successfully built lexicalrichness


In [3]:
import pandas as pd
import numpy as np

In [4]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("M4 GPU (Metal) is available!")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA GPU is available!")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU.")

GPU Available: True
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA GPU is available!


MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [20]:
df = pd.read_csv('../dataset/ai_human_text_dataset_2.csv')
df.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/ai_human_text_dataset_2.csv'

In [ ]:
df['text'] = df['text'].dropna().astype(str)

In [ ]:
## Exploratory Data Analysis for Text

def clean_text(text_df):

    # Lowercase the text
    text_df = text_df.str.lower()

    # Remove duplicates
    text_df = text_df.drop_duplicates()

    # Remove leading and trailing whitespace
    text_df = text_df.str.strip()

    # Remove punctuation
    text_df = text_df.str.replace(r'\d+\.\d+|\d+', '', regex=True)

    # Long text truncation
    text_df = text_df.str.slice(0, 500)
    
    return text_df


text_df = df[['text', 'label']]

text_df['text'] = clean_text(text_df['text'])

text_df.head(5)

In [ ]:
text_df = text_df.dropna().reset_index(drop=True)
text_df.shape

In [ ]:
# Lemmatization
import spacy

from spacy.lang.en.stop_words import STOP_WORDS

nlp = spacy.load('en_core_web_sm')

def preprocess_text(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        if token.text not in STOP_WORDS and not token.is_punct and not token.is_space:
            tokens.append(token.lemma_)
            
    return ' '.join(tokens)

text_df['text'] = text_df['text'].apply(preprocess_text)
df['lemmatized_text'] = text_df['text']

text_df.head(5)

In [ ]:
# Word clouds
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

def generate_cloud(text, title, ax):
    # Combine all text into a single string
    combined_text = ' '.join(text).split()
    
    word_counts = Counter(combined_text)
    
    # Generate word cloud
    wordcloud = WordCloud(width=800, height=400, 
                          background_color='white', 
                          colormap='viridis', 
                          max_words=100).generate_from_frequencies(word_counts)
    
    img_array = np.array(wordcloud.to_image())
    
    # Display the word cloud
    ax.imshow(img_array, interpolation='bilinear')
    ax.set_title(title, fontsize=20)
    ax.axis('off')
    
# Create figure
fig, axes = plt.subplots(1,2, figsize=(20, 10))    

# Generate word clouds for both classes
generate_cloud(text_df[text_df['label'] == 'human']['text'], 'Human Texts', axes[0])
generate_cloud(text_df[text_df['label'] == 'ai']['text'], 'AI-generated Texts', axes[1])

In [ ]:
# N-gram Analysis
from sklearn.feature_extraction.text import CountVectorizer

def get_top_n_words(corpus, n=10):
    # Converts text into a grid of numbers based on word frequency and creates a dictionary of all unique words found in the text
    vec = CountVectorizer(ngram_range=(2,2), stop_words='english').fit(corpus)
    
    # Counts how many times each of those words appears in every row of the data.
    bag_of_words = vec.transform(corpus)
    
    sum_words = bag_of_words.sum(axis=0)
    
    words_freq =  [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    
    return words_freq[:n]


human_words = get_top_n_words(text_df[text_df['label'] == 'human']['text'], n=10)
ai_words = get_top_n_words(text_df[text_df['label'] == 'ai']['text'], n=10)

print("Top 10 bigrams in Human-written texts:", human_words)
print("Top 10 bigrams in AI-generated texts:", ai_words)

In [ ]:
# TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf_vectorizer.fit_transform(df['lemmatized_text'].dropna())

tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

tfidf_df.head(5)

In [ ]:
# Complexity Metrics

from textstat import flesch_reading_ease, flesch_kincaid_grade
from lexical_diversity import lex_div as ld

df['text'] = df['text'].fillna('')
df['lemmatized_text'] = df['lemmatized_text'].fillna('')

df['flesch_reading_ease'] = df['text'].apply(flesch_reading_ease) #
df['flesch_kincaid_grade'] = df['text'].apply(flesch_kincaid_grade)

# TTR (Type-Token Ratio)
df['ttr'] = df['lemmatized_text'].apply(lambda x: ld.ttr(x.split()))

df.head(5)

In [ ]:
# Sentence Embeddings

from sentence_transformers import SentenceTransformer
import pandas as pd

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df['text'].fillna('').tolist())

embeddings_df = pd.DataFrame(embeddings, columns=[f'embedding_{i}' for i in range(embeddings.shape[1])])

embeddings_df.head(5)

In [ ]:
df['text'] = df['text'].replace([np.nan, None], " ") # Handle both types of nulls
df['text'] = df['text'].astype(str)
texts = df['text'].to_list()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load a pre-trained LLM
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

def get_llm_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to GPU
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0][0].cpu().numpy() # .cpu() moves result back for numpy

llm_embeddings_df = pd.DataFrame(df['text'].apply(get_llm_embeddings).tolist(), columns=[f'llm_embedding_{i}' for i in range(768)])

llm_embeddings_df.head(5)

In [ ]:
# Punctuation Frequency Features
import string
import pandas as pd

def get_punctuation_features(text):
    if not isinstance(text, str) or len(text) == 0:
        return {p: 0.0 for p in ['!', '?', '.', ',', ';', ':']}
    
    # Define the specific marks we want to track
    target_marks = ['!', '?', '.', ',', ';', ':']
    features = {}
    
    for mark in target_marks:
        # Calculate frequency: (Count of mark / Total characters)
        features[f'count_{mark}'] = text.count(mark) / len(text)
    
    return features

# Example Usage
punc_df = df['text'].apply(get_punctuation_features).apply(pd.Series)

punc_df.head(5)

In AI detection, Perplexity and Burstiness are the two most powerful structural signals. They help separate human "erratic" writing from AI "mechanical" consistency.


Perplexity measures how "surprised" a language model is by your text.

- Low Perplexity: The text follows a predictable pattern. (Classic AI trait).

- High Perplexity: The word choice is random or creative. (Classic Human trait)

In [ ]:
# Perplexity Calculation
import math
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model = model.to(device)
model.eval()

def get_perplexity(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = enc["input_ids"].to(device)
    with torch.inference_mode():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss.item()  # average negative log-likelihood per token
    return math.exp(loss)
    

perplexity_df = df['text'].apply(lambda x: pd.Series({'perplexity': get_perplexity(x)}))

perplexity_df.head(10)

# print(get_perplexity("This is a sample text to calculate perplexity."))

Burstiness measures the variation in sentence structure and length throughout a document.

- Low Burstiness: All sentences are roughly the same length (e.g., 15-20 words). (AI style).

- High Burstiness: Short, punchy sentences mixed with long, complex ones. (Human style).

In [ ]:
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')

In [ ]:
# Burstiness Calculation

import numpy as np
import nltk

def get_burstiness(text):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= 1:
        return 0.0
    
    # Calculate word count for each sentence
    sentence_lengths = [len(s.split()) for s in sentences]
    
    # Burstiness = Standard Deviation of sentence lengths
    # Higher std dev = More "bursty" (Human)
    return np.std(sentence_lengths)


burstiness_df = df['text'].apply(lambda x: pd.Series({'burstiness': get_burstiness(x)}))

burstiness_df.head(5)


In [ ]:
nltk.download('stopwords')

In [ ]:
# Stop word ratio
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def get_stopword_ratio(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0.0
    
    words = text.lower().split()
    
    if len(words) == 0:
        return 0.0
    
    # Count how many words in the text are in the NLTK stopword list
    stop_count = len([w for w in words if w in stop_words])
    
    # Return the ratio (0.0 to 1.0)
    return stop_count / len(words)

stopword_ratio_df = df['text'].apply(lambda x: pd.Series({'stopword_ratio': get_stopword_ratio(x)}))

stopword_ratio_df.tail(5)

In [ ]:
features_df = pd.concat([
    df[['label', 'flesch_kincaid_grade', 'flesch_reading_ease', 'ttr']].reset_index(drop=True), 
    # tfidf_df.reset_index(drop=True), 
    punc_df.reset_index(drop=True),
    burstiness_df.reset_index(drop=True),
    stopword_ratio_df.reset_index(drop=True),
    perplexity_df.reset_index(drop=True),
    # llm_embeddings_df.reset_index(drop=True),
    embeddings_df.reset_index(drop=True),
], axis=1)

features_df = features_df.fillna(0)

features_df.head(5)

In [ ]:
# PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df.drop(columns=['label']))

pca = PCA(n_components=0.95)  # Retain 95% variance
pca_features = pca.fit_transform(scaled_features)

print(pca_features.shape)

In [ ]:
# PCA slot

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df.drop(columns=['label']))

pca = PCA(n_components=0.95)  # Retain 95% variance
pca_features = pca.fit_transform(scaled_features)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=pca_features[:,0], y=pca_features[:,1], hue=features_df['label'], palette='viridis')
plt.title('PCA of Text Features')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

In [ ]:
# LDA - Finds directions that actually maximise class separations
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

lda = LDA(n_components=1)
lda_features = lda.fit_transform(scaled_features, features_df['label'])

plt.figure(figsize=(10, 5))
sns.kdeplot(lda_features[features_df['label'] == 'ai', 0], label='AI', fill=True)
sns.kdeplot(lda_features[features_df['label'] == 'human', 0], label='Human', fill=True)
plt.title('LDA Distribution: AI vs Human')
plt.xlabel('LD Component 1')
plt.legend()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Initialize t-SNE
# perplexity: roughly the number of neighbors each point considers (try 30-50)
# learning_rate: usually between 10 and 1000
tsne = TSNE(n_components=2, perplexity=40, random_state=42, init='pca', learning_rate='auto')

# 2. Fit and transform your scaled features
# X_scaled should contain your embeddings + burstiness + perplexity + punc
X_tsne = tsne.fit_transform(pca_features)

# 3. Create a DataFrame for easy plotting
tsne_df = pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2'])
tsne_df['label'] = df['label']  # Your 'AI' vs 'Human' labels

# 4. Plotting
plt.figure(figsize=(10, 7))
sns.scatterplot(data=tsne_df, x='tsne_1', y='tsne_2', hue='label', alpha=0.6)
plt.title("t-SNE Visualization: AI vs Human Text")
plt.show()

In [ ]:
# Machine Learning Models

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB # Almost normally distributed dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import  LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
features_df['label'] = label_encoder.fit_transform(features_df['label'])

# Train test split
X = pca_features
y = features_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=True, eval_metric='logloss')
}

In [ ]:
# Train and evaluate each model
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
        
    # Confusion Metrix
    # sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    # print("\n")

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'Roc-Auc Curve': roc_auc_score(y_test, y_prob)
    })
    
result_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)

result_df

In [ ]:
# Neural networks
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

## Early Stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Model 1: Simple Neural Network
model1 = Sequential([
    Dense(32, activation='relu', input_dim=(X_train.shape[1])), 
    Dense(1, activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history1 = model1.fit(X_train, y_train, 
                      validation_data=(X_test, y_test), 
                      epochs=50, 
                      batch_size=32, 
                      callbacks=[early_stop],
                      verbose=1)

# Model 2: Deeper Neural Network
# Sequential means you are building a neural network layer by layer, from start to finish.
# model2 = Sequential([
#     Dense(32, activation='relu', input_dim=(X_train.shape[1])), 
#     Dropout(0.3),
#     Dense(16, activation='relu'),
#     Dense(1, activation='sigmoid')
# ])

# model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# history2 = model2.fit(X_train, y_train, 
#                       validation_data=(X_test, y_test), 
#                       epochs=50, 
#                       batch_size=32, 
#                       callbacks=[early_stop],
#                       verbose=1)

# # Model 3: Wider Neural Network
# model3 = Sequential([
#     Dense(64, activation='relu', input_dim=(X_train.shape[1])), 
#     Dense(32, activation='relu'),
#     Dense(16, activation='relu'),
#     Dense(1, activation='sigmoid')
# ])

# model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# history3 = model3.fit(X_train, y_train, 
#                       validation_data=(X_test, y_test), 
#                       epochs=50, 
#                       batch_size=32, 
#                       callbacks=[early_stop],
#                       verbose=1)


In [ ]:
## Metric
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Function
def evaluate_model(model, X_test, y_test, name):
    y_pred = (model.predict(X_test) > 0.5).astype("int32")
    y_prob = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    return {
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Roc-Auc Curve': roc_auc
    }
    
# Evaluate
nn_results = []
nn_results.append(evaluate_model(model1, X_test, y_test, 'Simple NN'))
# nn_results.append(evaluate_model(model2, X_test, y_test, 'Deeper NN'))
# nn_results.append(evaluate_model(model3, X_test, y_test, 'Wider NN'))

nn_results_df = pd.DataFrame(nn_results).sort_values(by='Roc-Auc Curve', ascending=False)

nn_results_df